# Bronze to Silver Data Transformation (Sales)

All errors corrected in this notebook were identified in the bronze overview notebooks (Sales_2017_bronze, Sales_2018_bronze, Sales_2019_bronze, Sales_2020_bronze).

**Read files**

In [0]:
path = "/Volumes/sales_lakehouse/bronze/bronze_volume/Sales_2017.csv"


sales_2017_uncleaned = (spark.read
      .format("csv")
      .option("header", "true")
      .option("sep", "\t")          
      .option("inferSchema", "true")
      .option("mode", "PERMISSIVE")
      .load(path))

display(sales_2017_uncleaned.limit(5))

In [0]:
path = "/Volumes/sales_lakehouse/bronze/bronze_volume/Sales_2018.csv"


sales_2018_uncleaned = (spark.read
      .format("csv")
      .option("header", "true")
      .option("sep", "\t")          
      .option("inferSchema", "true")
      .option("mode", "PERMISSIVE")
      .load(path))

display(sales_2018_uncleaned.limit(5))

In [0]:
path = "/Volumes/sales_lakehouse/bronze/bronze_volume/Sales_2019.csv"


sales_2019_uncleaned = (spark.read
      .format("csv")
      .option("header", "true")
      .option("sep", "\t")          
      .option("inferSchema", "true")
      .option("mode", "PERMISSIVE")
      .load(path))

display(sales_2019_uncleaned.limit(5))

In [0]:
path = "/Volumes/sales_lakehouse/bronze/bronze_volume/Sales_2020.csv"


sales_2020_uncleaned = (spark.read
      .format("csv")
      .option("header", "true")
      .option("sep", "\t")          
      .option("inferSchema", "true")
      .option("mode", "PERMISSIVE")
      .load(path))

display(sales_2020_uncleaned.limit(5))

**Remove the extra zero from the ProductKeys in the Sales2017 DataFrame (data entry error)**

In [0]:
from pyspark.sql.functions import col, when

display(sales_2017_uncleaned.filter(col("ProductKey") > 999))

In [0]:
sales_2017_uncleaned = sales_2017_uncleaned.withColumn(
    "ProductKey",
    when(col("ProductKey") > 999, (col("ProductKey") / 10).cast("int"))
    .otherwise(col("ProductKey"))
)

In [0]:
display(sales_2017_uncleaned.filter(col("ProductKey") > 999))

**We union all the Sales tables into a single DataFrame.**

In [0]:
sales = (
    sales_2017_uncleaned
    .unionByName(sales_2018_uncleaned)
    .unionByName(sales_2019_uncleaned)
    .unionByName(sales_2020_uncleaned)

)

display(sales.limit(5))

In [0]:
sales.printSchema()

In [0]:
row_count = sales.count()
col_count = len(sales.columns)
print(f"number of rows: {row_count}, \nnumber of columns : {col_count}")

**Change Date Format**

In [0]:
from pyspark.sql.functions import regexp_replace, to_date

sales = sales.withColumn(
    "OrderDate",
    regexp_replace(col("OrderDate"), "^[A-Za-z]+, ", "")
)

sales = sales.withColumn(
    "OrderDate",
    to_date(col("OrderDate"), "MMMM d, yyyy")
)

display(sales.limit(5))

**Check for same SalesOrderNumber between the different dates**

In [0]:
from pyspark.sql.functions import countDistinct

display(sales.groupBy("SalesOrderNumber") \
    .agg(countDistinct("OrderDate").alias("distinct_dates")) \
    .filter("distinct_dates > 1"))

**Remove the "$" and the "," from the Unit Price, Cost and Sales columns**

In [0]:
sales = sales.withColumn("Unit Price", regexp_replace(col("Unit Price"), r"\$", "")) \
             .withColumn("Cost", regexp_replace(col("Cost"), r"\$", "")) \
             .withColumn("Sales", regexp_replace(col("Sales"), r"\$", ""))

In [0]:
from pyspark.sql.types import DecimalType

cols = ["Unit Price", "Cost", "Sales"]

for c in cols:
    sales = sales.withColumn(
        c,
        regexp_replace(col(c), "[$,]", "").cast(DecimalType(10,2))
    )

In [0]:
display(sales.limit(5))

In [0]:
display(sales)

**Change column names to lower case and snake case format**

In [0]:
sales = sales.withColumnRenamed("SalesOrderNumber", "sales_order_number") \
             .withColumnRenamed("OrderDate", "order_date") \
             .withColumnRenamed("ProductKey", "product_key") \
             .withColumnRenamed("ResellerKey", "reseller_key") \
             .withColumnRenamed("EmployeeKey", "employee_key") \
             .withColumnRenamed("SalesTerritoryKey", "sales_territory_key") \
             .withColumnRenamed("Quantity", "quantity") \
             .withColumnRenamed("Unit Price", "unit_price") \
             .withColumnRenamed("Cost", "cost") \
             .withColumnRenamed("Sales", "sales")

In [0]:
display(sales.limit(5))

**Check for data consistency**

In [0]:
display(sales.groupBy("sales_order_number") \
  .agg(countDistinct("reseller_key").alias("reseller_count")) \
  .filter("reseller_count > 1"))

In [0]:
display(sales.groupBy("sales_order_number")
.agg(countDistinct("employee_key").alias("reseller_count"))
.filter("reseller_count > 1"))

In [0]:
display(sales.groupBy("sales_order_number")
.agg(countDistinct("sales_territory_key").alias("reseller_count"))
.filter("reseller_count > 1"))

**Create Delta Table Schema**

In [0]:
%sql
DROP TABLE sales_lakehouse.silver.sales

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS sales_lakehouse.silver.sales (
    sales_order_number string,
    order_date date,
    product_key int,
    reseller_key int,
    employee_key int,
    sales_territory_key int,
    quantity int,
    unit_price decimal(10,2),
    sales decimal(10,2),
    cost decimal(10,2)
)
USING DELTA
""")

In [0]:
sales.write.format("delta").mode("overwrite").saveAsTable("sales_lakehouse.silver.sales")

In [0]:
display(sales)

In [0]:

sales.count()

**Validation Query**

In [0]:
%sql
SELECT COUNT(*) AS total_rows
FROM sales_lakehouse.silver.sales;

In [0]:
%sql
SELECT *
FROM sales_lakehouse.silver.sales;

In [0]:
%sql
Select *
from sales_lakehouse.silver.sales
where order_date = "2018-05-08" 